In [ ]:
!pip -q install scikit-learn

In [ ]:
import os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import random
import numpy as np

random.seed(42); np.random.seed(42)

vocab = {
    "invoice":  "invoice amount total tax payment due bill customer quantity price gst receipt paid balance order item subtotal".split(),
    "resume":   "experience skills education project university degree responsibilities worked developed team python java internship certification objective".split(),
    "research": "abstract method results dataset model experiment accuracy proposed analysis baseline algorithm training evaluation paper approach".split(),
    "legal":    "agreement party clause hereby liability terms jurisdiction contract court rights obligation notice breach indemnity witness".split(),
    "news":     "government minister announced police city report market match players election said yesterday officials public".split(),
}
common = "the of and to in is for on with as by at from this that are was be".split()
headers = {"invoice": "invoice no", "resume": "curriculum vitae", "research": "abstract",
           "legal": "this agreement", "news": "breaking news"}
classes = list(vocab)

def make_page(cls, first=False):
    n = random.randint(50, 80)
    words = [random.choice(vocab[cls]) if random.random() < 0.42 else random.choice(common) for _ in range(n)]
    if random.random() < 0.6:                                    # noise words from another class
        words += random.choices(vocab[random.choice(classes)], k=14)
    random.shuffle(words)
    if first and random.random() < 0.8:                          # most first pages carry a header
        words = headers[cls].split() + words
    return " ".join(words)

def make_stream(n_docs):
    """Returns pages, per-page class, per-page boundary flag (1 = first page of a document), doc ids."""
    pages, cls_lab, bnd, doc_id = [], [], [], []
    for d in range(n_docs):
        c = random.choice(classes)
        for j in range(random.randint(2, 5)):
            pages.append(make_page(c, first=(j == 0)))
            cls_lab.append(c); bnd.append(1 if j == 0 else 0); doc_id.append(d)
    return pages, cls_lab, bnd, doc_id

In [ ]:
# ---------- Part A: classify every page into a document type ----------
train_pages, train_cls, _, _ = make_stream(300)
test_pages, test_cls, _, _ = make_stream(100)
print("Train pages:", len(train_pages), "| Test pages:", len(test_pages), "| Classes:", classes)
print("\nSample page (invoice):", make_page("invoice", first=True)[:150], "...")

In [ ]:
nb = make_pipeline(TfidfVectorizer(), MultinomialNB()).fit(train_pages, train_cls)
svm = make_pipeline(TfidfVectorizer(sublinear_tf=True), LinearSVC()).fit(train_pages, train_cls)

for name, m in [("Naive Bayes", nb), ("Linear SVM", svm)]:
    pred = m.predict(test_pages)
    print(f"===== {name} | accuracy = {accuracy_score(test_cls, pred):.4f} =====")
    print(classification_report(test_cls, pred))

ConfusionMatrixDisplay.from_estimator(svm, test_pages, test_cls, xticks_rotation=45, cmap="Blues")
plt.title("Page classification - Linear SVM"); plt.show()

In [ ]:
# ---------- Part B: separate a mixed stream of pages into documents ----------
stream, true_cls, true_bnd, true_doc = make_stream(40)
print("Pages in the scanned stream:", len(stream), "| true documents:", max(true_doc) + 1)

pred_cls = svm.predict(stream)
vec = svm.named_steps["tfidfvectorizer"]
X = vec.transform(stream)
sim_prev = np.array([0.0] + [cosine_similarity(X[i], X[i-1])[0, 0] for i in range(1, len(stream))])
header_flag = np.array([any(p.startswith(h) for h in headers.values()) for p in stream])

def prf(y_true, y_pred):
    y_true, y_pred = np.array(y_true[1:]), np.array(y_pred[1:])       # first page is always a boundary
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    p = tp / (tp + fp) if tp + fp else 0
    r = tp / (tp + fn) if tp + fn else 0
    return round(p, 3), round(r, 3), round(2 * p * r / (p + r), 3) if p + r else 0

# Rule 1: boundary when predicted class changes
b1 = [1] + [int(pred_cls[i] != pred_cls[i-1]) for i in range(1, len(stream))]
# Rule 2: class change OR header detected on the page OR very low similarity to previous page
b2 = [1] + [int(pred_cls[i] != pred_cls[i-1] or header_flag[i] or sim_prev[i] < 0.25) for i in range(1, len(stream))]

res = pd.DataFrame([("Class change only", *prf(true_bnd, b1)),
                    ("Class change + header + similarity", *prf(true_bnd, b2))],
                   columns=["method", "precision", "recall", "F1"])
display(res)

In [ ]:
# ---------- Part C: write the separated documents to files ----------
boundaries = b2
out_dir = "separated_documents"
shutil.rmtree(out_dir, ignore_errors=True); os.makedirs(out_dir)

docs, cur = [], []
for i, b in enumerate(boundaries):
    if b and cur:
        docs.append(cur); cur = []
    cur.append(i)
docs.append(cur)

summary = []
for k, pages in enumerate(docs, 1):
    label = pd.Series(pred_cls[pages]).mode()[0]                       # majority vote of page classes
    with open(f"{out_dir}/doc_{k:02d}_{label}.txt", "w") as f:
        f.write("\n\n----- page break -----\n\n".join(stream[i] for i in pages))
    summary.append((k, f"{pages[0]+1}-{pages[-1]+1}", len(pages), label))

display(pd.DataFrame(summary, columns=["doc", "pages", "num_pages", "predicted type"]).head(12))
print("Detected documents:", len(docs), "| True documents:", max(true_doc) + 1)
print("Files written:", sorted(os.listdir(out_dir))[:5], "...")

In [ ]:
# ---------- Visualise true vs predicted document boundaries ----------
fig, ax = plt.subplots(figsize=(13, 2.2))
ax.vlines([i for i, b in enumerate(true_bnd) if b], 0.55, 1, colors="green", label="true boundary")
ax.vlines([i for i, b in enumerate(boundaries) if b], 0, 0.45, colors="crimson", label="predicted boundary")
ax.set_yticks([]); ax.set_xlabel("page number in scanned stream"); ax.legend(loc="upper right", ncol=2)
plt.title("Document boundary detection"); plt.show()